In [6]:
using ITensors
using LinearAlgebra
using Plots
using StatsBase
using DelimitedFiles

In [7]:
ITensors.space(::SiteType"Replica") = 6

uq = (readdlm("tensor_networks/T2_q1.csv",','))
function ITensors.op(::OpName"Transfer",::SiteType"Replica", s1::Index, s2::Index)
  return itensor(Float64,reduce(vcat,uq), s2', s1', s2, s1)
end

ITensors.state(::StateName"FUp", ::SiteType"Replica") = [1.,1., 1., 0., 0., 1.,]
ITensors.state(::StateName"FDown", ::SiteType"Replica") = [1.,0., 0., 1., 1., 1.]

In [4]:
tmax = 15
N = 20
NA = 3
purity_theta = []
entropy_proj_theta = []


for th in [0.4,]
  ITensors.state(::StateName"Theta", ::SiteType"Replica") = [cos(th/2)^4,cos(th/2)^2*sin(th/2)^2,cos(th/2)^2*sin(th/2)^2,cos(th/2)^2*sin(th/2)^2,cos(th/2)^2*sin(th/2)^2,sin(th/2)^4]
  purity_t = [] 
  entropy_proj_t = []

  sites = siteinds("Replica",N)
  psi = productMPS(Float64,sites,"Theta")
  bnd_purity = productMPS(Float64,sites,n-> n<=NA ? "FDown" : "FUp")
  
  for t in 0:tmax
    push!(purity_t,inner(bnd_purity,psi))

    val = 0.
    for k in 0:NA
      kk = 2*pi*k/(NA+1)
      ITensors.state(::StateName"FDownk", ::SiteType"Replica") = [1.,0., 0., exp(1im*kk), exp(-1im*kk), 1.]
      bnd = productMPS(Complex{Float64},sites,n-> n<=NA ? "FDownk" : "FUp")
      val+=inner(bnd,psi)
    end
    push!(entropy_proj_t,(abs(val))/(NA+1))
    
    os = [ ("Transfer",2*i-1,2*i) for i=1:Integer(N/2) ]
    psi = apply(ops(os,sites),psi;cutoff=1e-10)
    os = [ ("Transfer",2*i,mod(2*i,N)+1) for i=1:Integer(N/2) ]
    psi = apply(ops(os,sites),psi;cutoff=1e-10)
  end 
  push!(purity_theta,purity_t)
  push!(entropy_proj_theta,entropy_proj_t)
end 
